In [0]:
import os 
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt


1. Isolation Forest

In [0]:
# Get the current directory
current_directory = os.getcwd()

# Get the source file path
file_path = f'{current_directory}/datasets/processed'
file = f"{file_path}/processed_training_data.csv"
df = pd.read_csv(file, header=0)
df

In [0]:
X_train = df.drop(columns=['sus', 'evil'])
y_train = df['evil']

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest

numeric_features = ["stackAddresses_len", "stackAddresses_jump_std","argsNum"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", RobustScaler(), numeric_features)
    ],
    remainder="passthrough"
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("iforest", IsolationForest(
            n_estimators=300,
            contamination=0.0001,
            max_features=0.7,
            random_state=2000
        ))
    ]
)



In [0]:
# Perform fit on X and returns labels for X.
y_pred = model.fit_predict(X_train, y_train)

# Returns -1 for outliers and 1 for inliers.
# change -1 to 0:
y_pred = np.where(y_pred == -1, 1, 0)

# Get unique values and their counts in y_pred
unique_values, counts = np.unique(y_pred, return_counts=True)
print(f"Unique values: {unique_values}")
print(f"Counts of each value: {counts}")

# Get unique values and their counts in y_train
unique_values, counts = np.unique(y_train, return_counts=True)
print(f"Unique values: {unique_values}")
print(f"Counts of each value: {counts}")

In [0]:
cm = confusion_matrix(y_train, y_pred, labels=[0,1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=[0,1])
disp.plot()
plt.show()


In [0]:
anomaly_scores = model.score_samples(X_train)

anomaly_scores = -anomaly_scores  # Invert scores so that higher values indicate more anomalous instances
plt.hist(anomaly_scores, bins=100)
plt.title("Isolation Forest Anomaly Score Distribution")
plt.show()